# Guardar y recuperar un modelo entrenado en scikit-learn

En este mini-cuaderno vamos a entrenar rápidamente un modelo de clasificación y luego veremos cómo guardarlo en un archivo para poder reutilizarlo más adelante sin repetir el entrenamiento.

La parte de entrenamiento será breve, porque es un tema que ya vimos en cuadernos anteriores. En este caso, el objetivo principal es trabajar con la persistencia del modelo.

## 1. Importamos las librerías necesarias

Usaremos un dataset de ejemplo incluido en `scikit-learn`, entrenaremos un modelo sencillo y luego guardaremos el modelo con `joblib`.

In [25]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

## 2. Cargamos un dataset de ejemplo

Usaremos el dataset Iris, que contiene mediciones de flores y la especie correspondiente. Es un problema de clasificación, porque el modelo debe predecir una categoría.

In [26]:
# Cargamos el dataset Iris
datos = load_iris()

# Variables de entrada
X = datos.data

# Variable objetivo
y = datos.target

# Mostramos información básica
print("Cantidad de muestras:", X.shape[0])
print("Cantidad de variables:", X.shape[1])
print("Clases:", datos.target_names)

Cantidad de muestras: 150
Cantidad de variables: 4
Clases: ['setosa' 'versicolor' 'virginica']


## 3. Separamos los datos en entrenamiento y prueba

Dividimos los datos para entrenar el modelo con una parte y evaluarlo con otra. Esto nos permite estimar cómo se comporta el modelo con datos que no vio durante el entrenamiento.

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Datos de entrenamiento:", X_train.shape)
print("Datos de prueba:", X_test.shape)

Datos de entrenamiento: (120, 4)
Datos de prueba: (30, 4)


## 4. Creamos y entrenamos el modelo

Entrenaremos un `RandomForestClassifier`. En este cuaderno no nos detendremos demasiado en el algoritmo, porque el foco está en guardar y recuperar el modelo entrenado.

In [28]:
modelo = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

modelo.fit(X_train, y_train)

print("Modelo entrenado correctamente.")

Modelo entrenado correctamente.


## 5. Evaluamos rápidamente el modelo

Antes de guardar el modelo, conviene comprobar que funciona correctamente. Para eso hacemos predicciones sobre el conjunto de prueba y calculamos algunas métricas.

In [29]:
y_pred = modelo.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print()
print(classification_report(y_test, y_pred, target_names=datos.target_names))

Accuracy: 0.9

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.82      0.90      0.86        10
   virginica       0.89      0.80      0.84        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30



## 6. Guardamos el modelo entrenado

Hasta este punto, el modelo ya fue entrenado con los datos de entrenamiento. Eso significa que el objeto `modelo` ya contiene la información que aprendió durante el proceso de entrenamiento.

En algunos algoritmos, esa información puede aparecer como coeficientes, pesos o parámetros internos. En otros modelos, como los árboles de decisión o los bosques aleatorios, lo aprendido no se guarda exactamente como “pesos”, sino como estructuras internas del modelo.

Por eso, en `scikit-learn` solemos hablar de **guardar el modelo entrenado completo**, no solamente sus pesos.

Guardar el modelo es útil porque nos permite reutilizarlo más adelante sin volver a ejecutar el entrenamiento. Esto puede ser importante cuando:

* el entrenamiento tarda mucho tiempo;
* el dataset es grande;
* queremos usar el modelo en otra notebook;
* queremos llevar el modelo a una aplicación;
* queremos compartir el modelo entrenado con otra persona;
* queremos conservar una versión específica del modelo.

Para guardar modelos de `scikit-learn`, una herramienta muy usada es `joblib`. Esta librería permite guardar objetos de Python en archivos y luego recuperarlos.

En este caso, vamos a guardar el modelo entrenado en un archivo llamado:

```python
modelo_iris.pkl
```

La extensión `.pkl` suele usarse para archivos generados mediante serialización en Python. También podríamos usar una extensión como `.joblib`; lo importante es que luego recordemos con qué herramienta lo vamos a cargar.

Una vez guardado el archivo, podremos recuperar el modelo más adelante con `joblib.load()` y usarlo directamente para hacer predicciones, sin volver a ejecutar `modelo.fit()`.


In [30]:
joblib.dump(modelo, "modelo_iris.pkl")

print("Modelo guardado en el archivo modelo_iris.pkl")

Modelo guardado en el archivo modelo_iris.pkl


## 7. Simulamos que empezamos de nuevo

Para comprobar que realmente podemos recuperar el modelo desde el archivo, vamos a eliminar la variable `modelo` de la memoria.

Esto simula una situación en la que cerramos el entorno y luego queremos volver a usar el modelo guardado.

In [31]:
del modelo

print("La variable modelo fue eliminada de la memoria.")

La variable modelo fue eliminada de la memoria.


## 8. Cargamos el modelo guardado

Ahora recuperamos el modelo desde el archivo `.pkl`. No volveremos a entrenarlo: simplemente lo cargamos y lo usamos.

In [32]:
modelo_recuperado = joblib.load("modelo_iris.pkl")

print("Modelo recuperado correctamente.")

Modelo recuperado correctamente.


## 9. Usamos el modelo recuperado para predecir

El modelo recuperado ya conserva lo que aprendió durante el entrenamiento. Podemos usarlo directamente con `.predict()`.

In [33]:
y_pred_recuperado = modelo_recuperado.predict(X_test)

accuracy_recuperado = accuracy_score(y_test, y_pred_recuperado)

print("Accuracy del modelo recuperado:", accuracy_recuperado)

Accuracy del modelo recuperado: 0.9


## 10. Probamos el modelo con un dato nuevo

Ahora vamos a usar el modelo recuperado para clasificar una flor nueva. El dato debe tener el mismo formato que los datos usados para entrenar: cuatro valores numéricos.

In [34]:
# Ejemplo de una flor nueva
# Valores: largo sépalo, ancho sépalo, largo pétalo, ancho pétalo
flor_nueva = [[5.1, 3.5, 1.4, 0.2]]

prediccion = modelo_recuperado.predict(flor_nueva)

clase_predicha = datos.target_names[prediccion[0]]

print("Clase predicha:", clase_predicha)

Clase predicha: setosa


## 11. Otra opción: descargar el archivo

También podemos descargar el archivo `.pkl` a nuestra computadora. Esto es útil si queremos conservar una copia local.

In [35]:
from google.colab import files

files.download("modelo_iris.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 12. Idea importante: guardar el pipeline completo

Si el modelo utiliza pasos previos de preprocesamiento, como escalado, codificación de variables categóricas o imputación de valores faltantes, conviene guardar el `Pipeline` completo, no solamente el modelo final.

De esa manera, cuando lleguen datos nuevos, se aplicarán exactamente las mismas transformaciones usadas durante el entrenamiento.

In [36]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline(
    steps=[
        ("escalado", StandardScaler()),
        ("modelo", LogisticRegression(max_iter=1000))
    ]
)

pipeline.fit(X_train, y_train)

joblib.dump(pipeline, "pipeline_iris.pkl")

print("Pipeline completo guardado correctamente.")

Pipeline completo guardado correctamente.


## 13. Cargamos y usamos el pipeline completo

Cuando cargamos el pipeline, podemos usarlo directamente para predecir. El escalado se aplica automáticamente antes del modelo.

In [37]:
pipeline_recuperado = joblib.load("pipeline_iris.pkl")

prediccion_pipeline = pipeline_recuperado.predict(flor_nueva)

print("Clase predicha por el pipeline:", datos.target_names[prediccion_pipeline[0]])

Clase predicha por el pipeline: setosa


## Cierre

Guardar un modelo entrenado permite reutilizarlo sin repetir el entrenamiento. Esto es especialmente útil cuando el entrenamiento tarda mucho tiempo o cuando queremos usar el modelo en otro programa, otra notebook o una aplicación.

En Colab, si queremos conservar el archivo, debemos descargarlo, porque el entorno de trabajo puede reiniciarse y perder los archivos temporales.